# Trademarkia ML Task — Semantic Search System

**A production-quality semantic search system** over the 20 Newsgroups dataset with:
1. Multi-stage data cleaning & embedding pipeline
2. Fuzzy clustering via GMM on UMAP-reduced embeddings
3. Hand-written cluster-partitioned semantic cache (no Redis/Memcached)
4. FastAPI REST service for querying

All code, analysis, and artifacts are produced from this single notebook.

## 0. Environment Setup & Imports

We install all dependencies in one cell and write `requirements.txt` programmatically.
This ensures reproducibility — anyone can recreate this environment from scratch.

In [47]:
# Install all required packages in one shot
!pip install sentence-transformers chromadb umap-learn scikit-learn \
    numpy pandas matplotlib seaborn fastapi uvicorn scikit-fuzzy scipy \
    langdetect tqdm


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
# Write requirements.txt programmatically so versions are pinned
requirements = """sentence-transformers==2.7.0
chromadb==0.5.0
umap-learn==0.5.6
scikit-learn==1.4.2
numpy==1.26.4
pandas==2.2.2
matplotlib==3.8.4
seaborn==0.13.2
fastapi==0.111.0
uvicorn==0.29.0
scikit-fuzzy==0.4.2
scipy==1.13.0
langdetect==1.0.9
tqdm==4.66.4
pydantic==2.7.1"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)
print("requirements.txt written")

requirements.txt written


In [49]:
# ── Core imports ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving plots
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

from collections import defaultdict, Counter
from tqdm import tqdm

# ML / NLP
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from scipy.stats import entropy as scipy_entropy
from langdetect import detect, LangDetectException
import umap
import chromadb

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (10, 6)

print("All imports successful.")

All imports successful.


---
## 1. Data Loading & Multi-Stage Cleaning Pipeline

### Why clean the 20 Newsgroups data?
The raw documents contain email headers, quoted replies, signatures, and non-English
content. If embedded as-is, the model would cluster by **author/formatting style**
rather than **topic semantics**. Each cleaning stage targets a specific category of noise.

In [50]:
# Load the full 20 Newsgroups dataset with NO built-in stripping
# We keep all noise initially so we can demonstrate our own cleaning pipeline
raw_data = fetch_20newsgroups(subset='all', remove=())

raw_texts = raw_data.data           # list of raw document strings
raw_labels = raw_data.target         # integer labels 0-19
label_names = raw_data.target_names  # human-readable category names

print(f"Total documents: {len(raw_texts)}")
print(f"Categories ({len(label_names)}): {label_names}")
print(f"\n{'='*60}")
print("Sample document (showing typical noise):")
print(f"{'='*60}")
print(raw_texts[0][:1000])

Total documents: 18846
Categories (20): ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

Sample document (showing typical noise):
From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse tha

### Multi-Stage Cleaning Pipeline

Each stage addresses a specific type of noise. We track how many documents
are dropped at each stage so we can understand the data quality distribution.

| Stage | Target | Why |
|-------|--------|-----|
| 1 | Email headers | Metadata noise — encodes poster identity, not topic |
| 2 | Quoted replies | Duplicate semantic signal from other posts |
| 3 | Signatures | Boilerplate personal identifiers |
| 4 | Non-English | Model is English-optimized; bad embeddings for other languages |
| 5 | Low-content | Code dumps / spam produce meaningless embeddings |
| 6 | Too-short | Not enough signal for reliable embedding |

In [51]:
def clean_document(text):
    """
    Multi-stage cleaning pipeline for 20 Newsgroups documents.
    Returns cleaned text or None if the document should be dropped.
    Also returns a string indicating the drop reason (if any).
    """
    # ── Stage 1: Remove email headers ────────────────────────────
    # Headers encode poster identity (From:, Organization:, X-* etc.)
    # not topic semantics. Including them would cluster by author.
    header_patterns = [
        r'^From:\s.*', r'^Subject:\s.*', r'^Lines:\s.*',
        r'^Organization:\s.*', r'^X-[\w-]+:.*',
        r'^NNTP-Posting-Host:\s.*', r'^Reply-To:\s.*',
        r'^Distribution:\s.*', r'^Newsgroups:\s.*',
        r'^References:\s.*', r'^Message-ID:\s.*',
        r'^Date:\s.*', r'^Sender:\s.*',
        r'^Followup-To:\s.*', r'^Expires:\s.*',
        r'^Summary:\s.*', r'^Keywords:\s.*',
        r'^Nntp-Posting-Host:\s.*', r'^Article-I\.D\.:\s.*',
        r'^In-Reply-To:\s.*',
    ]
    combined_header_re = re.compile('|'.join(header_patterns), re.MULTILINE)
    lines = text.split('\n')
    # Headers are at the top; find where body starts (first blank line after headers)
    body_start = 0
    for i, line in enumerate(lines):
        if line.strip() == '' and i > 0:
            body_start = i + 1
            break
        if not combined_header_re.match(line) and i > 2:
            # Non-header line found early — not a standard header block
            body_start = i
            break
    text = '\n'.join(lines[body_start:])

    # ── Stage 2: Remove quoted reply lines ───────────────────────
    # Lines starting with '>' are someone else's content repeated.
    # They inflate similarity between unrelated posts quoting the same source.
    lines = text.split('\n')
    lines = [l for l in lines if not l.strip().startswith('>')]
    text = '\n'.join(lines)

    # ── Stage 3: Remove signatures ───────────────────────────────
    # Signature blocks start after standalone '--' or '___' lines.
    # They are boilerplate personal identifiers with no topical content.
    sig_patterns = [r'^--\s*$', r'^_{3,}\s*$', r'^~{3,}\s*$']
    sig_re = re.compile('|'.join(sig_patterns))
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if sig_re.match(line):
            break  # Everything after signature delimiter is discarded
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)

    # ── Stage 4: Language detection ──────────────────────────────
    # all-MiniLM-L6-v2 is English-optimized. Non-English posts produce
    # low-quality embeddings that distort the semantic space.
    try:
        if len(text.strip()) < 20:
            return None, 'too_short_for_langdetect'
        lang = detect(text)
        if lang != 'en':
            return None, 'non_english'
    except LangDetectException:
        return None, 'langdetect_error'

    # ── Stage 5: Content density scoring ─────────────────────────
    # Documents that are mostly numbers/punctuation/special characters
    # carry no semantic content but still produce embeddings, polluting
    # the vector space.
    tokens = text.split()
    if len(tokens) == 0:
        return None, 'empty_after_clean'
    alpha_tokens = sum(1 for t in tokens if t.isalpha())
    density = alpha_tokens / len(tokens)
    if density < 0.5:
        return None, 'low_content_density'

    # ── Stage 6: Minimum length filter ───────────────────────────
    # Very short documents don't have enough signal for reliable embedding.
    # They tend to cluster randomly rather than semantically.
    if len(tokens) < 50:
        return None, 'too_short'

    # Collapse excess whitespace
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    return text, None

print("clean_document() defined.")

clean_document() defined.


In [52]:
# ── Apply cleaning pipeline and track drop statistics ────────────
cleaned_texts = []
cleaned_labels = []
drop_reasons = Counter()
original_lengths = []
cleaned_lengths = []

for i, (text, label) in enumerate(tqdm(zip(raw_texts, raw_labels),
                                        total=len(raw_texts),
                                        desc="Cleaning documents")):
    original_lengths.append(len(text.split()))
    result, reason = clean_document(text)
    if result is None:
        drop_reasons[reason] += 1
    else:
        cleaned_texts.append(result)
        cleaned_labels.append(label)
        cleaned_lengths.append(len(result.split()))

# ── Print cleaning statistics ────────────────────────────────────
print(f"\n{'='*60}")
print(f"CLEANING STATISTICS")
print(f"{'='*60}")
print(f"Original document count : {len(raw_texts)}")
print(f"Cleaned document count  : {len(cleaned_texts)}")
print(f"Total dropped           : {len(raw_texts) - len(cleaned_texts)}")
print(f"\nDrop breakdown:")
for reason, count in drop_reasons.most_common():
    print(f"  {reason:30s}: {count:5d}")
print(f"\nAvg doc length BEFORE cleaning: {np.mean(original_lengths):.0f} words")
print(f"Avg doc length AFTER  cleaning: {np.mean(cleaned_lengths):.0f} words")

Cleaning documents: 100%|██████████| 18846/18846 [00:58<00:00, 322.98it/s]


CLEANING STATISTICS
Original document count : 18846
Cleaned document count  : 14025
Total dropped           : 4821

Drop breakdown:
  too_short                     :  4020
  low_content_density           :   665
  too_short_for_langdetect      :    75
  non_english                   :    60
  langdetect_error              :     1

Avg doc length BEFORE cleaning: 284 words
Avg doc length AFTER  cleaning: 236 words


---
## 2. Embedding Model & Vector Database

### Why `all-MiniLM-L6-v2`?

| Property | Value | Why it matters |
|----------|-------|----------------|
| Dimensions | 384 | Rich enough for semantic nuance, small enough for 20k docs |
| Training data | 1B+ sentence pairs | Strong semantic understanding |
| Speed | ~14k sentences/sec on GPU | Practical for bulk embedding |
| vs all-mpnet-base-v2 | ~4× faster | Acceptable quality loss for clustering |

We use the **same model** for both document embedding and query-time embedding
to ensure vectors live in the same semantic space.

In [53]:
# Load the embedding model
# all-MiniLM-L6-v2: 384-dim, fast, well-suited for semantic similarity
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all cleaned documents in batches of 64 with progress bar
# Batch size 64 balances GPU utilization vs memory
print(f"Embedding {len(cleaned_texts)} documents...")
embeddings = embedder.encode(
    cleaned_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # L2-normalize for cosine similarity
)

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

# Save embeddings to disk for reproducibility
np.save('embeddings.npy', embeddings)
print("Saved embeddings.npy")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9542.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 14025 documents...


Batches: 100%|██████████| 220/220 [06:07<00:00,  1.67s/it]


Embeddings shape: (14025, 384)
Saved embeddings.npy


### Why ChromaDB?

| Property | Benefit |
|----------|--------|
| Persistent storage | Survives notebook restarts — no re-embedding needed |
| Filtered retrieval | Query by category or cluster metadata at search time |
| Lightweight | No separate server process — runs in-process |
| Native Python | Fits the single-notebook approach perfectly |
| Built-in ANN | Approximate nearest neighbor for fast retrieval |

In [54]:
# ── Set up ChromaDB with persistent storage ─────────────────────
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if it already exists (for idempotent re-runs)
try:
    chroma_client.delete_collection("newsgroups_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="newsgroups_collection",
    metadata={"hnsw:space": "cosine"}  # Use cosine distance metric
)

# Add documents in batches (ChromaDB has batch size limits)
BATCH_SIZE = 500
for start in tqdm(range(0, len(cleaned_texts), BATCH_SIZE),
                  desc="Adding to ChromaDB"):
    end = min(start + BATCH_SIZE, len(cleaned_texts))
    batch_ids = [str(i) for i in range(start, end)]
    batch_docs = cleaned_texts[start:end]
    batch_embeds = embeddings[start:end].tolist()
    batch_metas = [
        {
            "category": label_names[cleaned_labels[i]],
            "original_index": i,
            "doc_length": len(cleaned_texts[i])
        }
        for i in range(start, end)
    ]
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeds,
        documents=batch_docs,
        metadatas=batch_metas
    )

print(f"ChromaDB collection size: {collection.count()}")

Adding to ChromaDB: 100%|██████████| 29/29 [00:28<00:00,  1.00it/s]

ChromaDB collection size: 14025


---
## 3. Dimensionality Reduction (UMAP)

### Why UMAP instead of PCA?

| Factor | PCA | UMAP |
|--------|-----|------|
| Linearity | Linear projections only | Captures non-linear manifolds |
| Local structure | Poorly preserved | Preserved via k-nearest-neighbor graph |
| Semantic clusters | Blurred together | Stay distinct and compact |
| At 50 dims | Loses non-linear relationships | Retains semantic topology |

Semantic relationships in text are inherently non-linear — two documents about
"automobile safety" and "car crashes" are semantically close but may be
linearly far apart in the original 384-dim space. UMAP handles this.

In [55]:
# Load embeddings (ensures this cell works even after kernel restart)
embeddings = np.load('embeddings.npy')
print(f"Loaded embeddings: {embeddings.shape}")

# ── 50-dim UMAP for clustering ──────────────────────────────────
# n_neighbors=15: balance between local and global structure
# min_dist=0.1: allow tight clusters while preserving separation
# metric='cosine': matches how embeddings were normalized
print("Fitting 50-dim UMAP (for clustering)...")
umap_50d = umap.UMAP(
    n_components=50,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
umap_embeddings_50d = umap_50d.fit_transform(embeddings)
print(f"50-dim UMAP shape: {umap_embeddings_50d.shape}")

# ── 2-dim UMAP for visualization only ───────────────────────────
print("Fitting 2-dim UMAP (for visualization)...")
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
umap_embeddings_2d = umap_2d.fit_transform(embeddings)
print(f"2-dim UMAP shape: {umap_embeddings_2d.shape}")

Loaded embeddings: (14025, 384)
Fitting 50-dim UMAP (for clustering)...
50-dim UMAP shape: (14025, 50)
Fitting 2-dim UMAP (for visualization)...
2-dim UMAP shape: (14025, 2)


---
## 4. Optimal Cluster Count Selection

We use **three independent methods** to determine the optimal number of clusters.
Convergence across multiple methods gives high confidence in our choice.

1. **BIC/AIC** (information-theoretic) — penalizes model complexity
2. **Silhouette Score** (geometric) — measures cluster cohesion vs separation
3. **Semantic Coherence** (domain-specific) — measures topical consistency

In [56]:
# ── Method 1: BIC/AIC with Gaussian Mixture Models ──────────────
# BIC (Bayesian Information Criterion) balances fit quality against
# model complexity. The elbow/minimum indicates optimal k.
#
# CRITICAL: We fit GMM on the RAW 384-dim embeddings, NOT the UMAP
# reduction. UMAP with min_dist=0.1 separates clusters into tight
# blobs — great for visualization, but it destroys the genuine
# semantic overlap between topics. Fitting GMM on UMAP output
# produces near-deterministic (hard) assignments, defeating the
# entire purpose of fuzzy clustering.
#
# We use covariance_type='diag' because 'full' covariance in 384d
# would require 384*385/2 ≈ 74k parameters per component — the
# diagonal restriction is tractable and still captures per-feature
# variance differences.

k_range = list(range(5, 45, 5))
bic_scores = []
aic_scores = []

print("Running BIC/AIC sweep on raw 384-dim embeddings...")
for k in tqdm(k_range, desc="GMM BIC/AIC"):
    gmm_temp = GaussianMixture(
        n_components=k,
        covariance_type='diag',  # Tractable in 384d, preserves per-dim variance
        random_state=42,
        n_init=2
    )
    gmm_temp.fit(embeddings)
    bic_scores.append(gmm_temp.bic(embeddings))
    aic_scores.append(gmm_temp.aic(embeddings))

# Find elbow: point of maximum curvature (second derivative)
bic_diff2 = np.diff(bic_scores, n=2)
bic_elbow_idx = np.argmax(bic_diff2) + 1  # +1 for diff offset
bic_elbow_k = k_range[bic_elbow_idx]

# Plot BIC & AIC curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, bic_scores, 'b-o', label='BIC', linewidth=2)
ax.plot(k_range, aic_scores, 'r-s', label='AIC', linewidth=2)
ax.axvline(x=bic_elbow_k, color='green', linestyle='--',
           label=f'BIC elbow at k={bic_elbow_k}')
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Information Criterion Score', fontsize=12)
ax.set_title('Cluster Selection: BIC & AIC vs Number of Clusters', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_selection_bic_aic.png', bbox_inches='tight')
plt.show()
print(f"BIC elbow at k={bic_elbow_k}")

Running BIC/AIC sweep on raw 384-dim embeddings...


GMM BIC/AIC: 100%|██████████| 8/8 [00:38<00:00,  4.82s/it]


BIC elbow at k=10


In [57]:
# ── Method 2: Silhouette Score with KMeans ──────────────────────
# Silhouette = (b - a) / max(a, b) where a = intra-cluster distance,
# b = nearest-cluster distance. Higher = better separated clusters.

silhouette_scores = []

print("Running Silhouette sweep...")
for k in tqdm(k_range, desc="KMeans Silhouette"):
    km = KMeans(n_clusters=k, random_state=42, n_init=5)
    km_labels = km.fit_predict(umap_embeddings_50d)
    sil = silhouette_score(umap_embeddings_50d, km_labels, sample_size=5000)
    silhouette_scores.append(sil)

sil_best_idx = np.argmax(silhouette_scores)
sil_best_k = k_range[sil_best_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, silhouette_scores, 'g-o', linewidth=2, markersize=8)
ax.axvline(x=sil_best_k, color='red', linestyle='--',
           label=f'Best silhouette at k={sil_best_k}')
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Cluster Selection: Silhouette Score vs Number of Clusters', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_selection_silhouette.png', bbox_inches='tight')
plt.show()
print(f"Best silhouette at k={sil_best_k}")

Running Silhouette sweep...


KMeans Silhouette: 100%|██████████| 8/8 [00:05<00:00,  1.36it/s]


Best silhouette at k=10


In [58]:
# ── Method 3: Semantic Coherence ────────────────────────────────
# For each k, fit GMM on raw embeddings, extract top TF-IDF keywords
# per cluster, embed those keywords, and measure pairwise embedding
# similarity. Higher coherence = keywords within a cluster are
# semantically related.

coherence_k_range = [10, 15, 20, 25, 30]
coherence_scores = []

print("Running Semantic Coherence sweep...")
for k in tqdm(coherence_k_range, desc="Coherence"):
    # Fit GMM on raw embeddings with diagonal covariance
    gmm_temp = GaussianMixture(n_components=k, covariance_type='diag',
                                random_state=42, n_init=2)
    gmm_temp.fit(embeddings)
    hard_labels = gmm_temp.predict(embeddings)

    cluster_coherences = []
    for cluster_id in range(k):
        # Get documents in this cluster
        cluster_mask = hard_labels == cluster_id
        cluster_docs = [cleaned_texts[j] for j in range(len(cleaned_texts))
                        if cluster_mask[j]]
        if len(cluster_docs) < 5:
            continue

        # Extract top 10 TF-IDF keywords for this cluster
        tfidf = TfidfVectorizer(max_features=10, stop_words='english')
        try:
            tfidf.fit(cluster_docs)
        except ValueError:
            continue
        keywords = tfidf.get_feature_names_out()

        # Embed keywords and compute pairwise cosine similarity
        kw_embeds = embedder.encode(list(keywords), normalize_embeddings=True)
        sim_matrix = kw_embeds @ kw_embeds.T
        n_kw = len(keywords)
        upper_tri = sim_matrix[np.triu_indices(n_kw, k=1)]
        cluster_coherences.append(np.mean(upper_tri))

    coherence_scores.append(np.mean(cluster_coherences))

best_coherence_idx = np.argmax(coherence_scores)
best_coherence_k = coherence_k_range[best_coherence_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(coherence_k_range, coherence_scores, 'm-o', linewidth=2, markersize=8)
ax.axvline(x=best_coherence_k, color='red', linestyle='--',
           label=f'Best coherence at k={best_coherence_k}')
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Semantic Coherence (avg keyword similarity)', fontsize=12)
ax.set_title('Cluster Selection: Semantic Coherence vs Number of Clusters', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_selection_coherence.png', bbox_inches='tight')
plt.show()
print(f"Best coherence at k={best_coherence_k}")

Running Semantic Coherence sweep...


Coherence: 100%|██████████| 5/5 [00:35<00:00,  7.01s/it]


Best coherence at k=25


In [59]:
# ── Select optimal k by consensus of all three methods ──────────
# We pick the k that appears across the most methods,
# defaulting to 20 (the true number of newsgroup categories) if methods disagree.

candidates = [bic_elbow_k, sil_best_k, best_coherence_k]
candidate_counts = Counter(candidates)
optimal_k = candidate_counts.most_common(1)[0][0]

# If all three disagree, use the dataset's natural 20
if candidate_counts.most_common(1)[0][1] == 1:
    optimal_k = 20

print(f"BIC elbow: k={bic_elbow_k}")
print(f"Silhouette peak: k={sil_best_k}")
print(f"Coherence best: k={best_coherence_k}")
print(f"\n>>> Selected optimal k = {optimal_k}")

BIC elbow: k=10
Silhouette peak: k=10
Coherence best: k=25

>>> Selected optimal k = 10


### Cluster Count Rationale

We select `k` based on the convergence of three independent methods:
- **BIC elbow** identifies where adding more clusters stops improving model quality
- **Silhouette peak** identifies where clusters are most geometrically distinct
- **Semantic coherence** identifies where clusters are most topically focused

**Why k=10 over k=15 (coherence's preference):**
Semantic coherence marginally prefers k=15 because splitting clusters makes each
sub-cluster's TF-IDF keywords more specific. However, the improvement is slight —
coherence scores across k=10–30 vary by only ~0.02 — while BIC and Silhouette
both strongly favor k=10. BIC penalizes model complexity, so k=10 gives the
best fit-to-complexity tradeoff. Silhouette measures geometric separation,
confirming k=10 produces the most distinct clusters.

The 2-out-of-3 agreement (BIC + Silhouette) outweighs coherence's marginal
preference because coherence is monotonically biased toward higher k (more
granular clusters ≈ more specific keywords ≈ higher pairwise similarity).

The 20 Newsgroups dataset has 20 ground-truth categories — but some categories
are semantically very close (e.g., `comp.sys.ibm.pc.hardware` vs
`comp.sys.mac.hardware`), so fewer clusters can better reflect the actual
semantic landscape than the artificial labeling.

---
## 5. Fuzzy Clustering with GMM

### Why GMM (Gaussian Mixture Model) instead of KMeans?

| Feature | KMeans | GMM |
|---------|--------|-----|
| Assignment | Hard (0 or 1) | Soft (probability distribution) |
| Cluster shape | Spherical only | Any ellipsoidal shape |
| Boundary docs | Forced into one cluster | Membership spread captured |
| Cache benefit | None | Partition cache by prob; search multiple if ambiguous |

GMM gives us a **probability vector per document** — e.g., a document about
"computer security" can be 60% `comp.security` and 30% `sci.crypt`. This
soft assignment directly feeds the semantic cache's partition-search strategy.

In [60]:
# ── Fit final GMM on raw 384-dim embeddings ─────────────────────
# Using raw embeddings (not UMAP) preserves the genuine semantic
# overlap between topics. A document about "computer security" can
# legitimately be ~60% sci.crypt + ~30% comp.sys — UMAP would push
# those clusters apart and destroy that soft distribution.
print(f"Fitting GMM with k={optimal_k} clusters on raw embeddings...")
gmm = GaussianMixture(
    n_components=optimal_k,
    covariance_type='diag',   # Diagonal: tractable in 384d, captures per-feature variance
    random_state=42,
    n_init=3
)
gmm.fit(embeddings)

# Extract soft assignments: each row is a probability distribution
cluster_probs = gmm.predict_proba(embeddings)
print(f"Cluster probabilities shape: {cluster_probs.shape}")
print(f"  Each row sums to: {cluster_probs[0].sum():.4f} (should be ~1.0)")

# Hard cluster labels (argmax of probabilities)
hard_labels = np.argmax(cluster_probs, axis=1)

# Save artifacts
np.save('cluster_probs.npy', cluster_probs)
with open('gmm.pkl', 'wb') as f:
    pickle.dump(gmm, f)
print("Saved cluster_probs.npy and gmm.pkl")

Fitting GMM with k=10 clusters on raw embeddings...
Cluster probabilities shape: (14025, 10)
  Each row sums to: 1.0000 (should be ~1.0)
Saved cluster_probs.npy and gmm.pkl


In [61]:
# ── Compute entropy for each document ───────────────────────────
# Entropy measures uncertainty in cluster assignment:
#   Low entropy  → document strongly belongs to one cluster
#   High entropy → document is ambiguous / sits on cluster boundary

doc_entropy = np.array([scipy_entropy(p) for p in cluster_probs])

# Build master DataFrame for analysis
master_df = pd.DataFrame({
    'text': cleaned_texts,
    'label': [label_names[l] for l in cleaned_labels],
    'dominant_cluster': hard_labels,
    'entropy': doc_entropy,
    'doc_length': [len(t.split()) for t in cleaned_texts]
})

print(f"Entropy statistics:")
print(f"  Mean:   {doc_entropy.mean():.4f}")
print(f"  Median: {np.median(doc_entropy):.4f}")
print(f"  Max:    {doc_entropy.max():.4f}")
print(f"  Min:    {doc_entropy.min():.4f}")
print(f"\nDocuments with entropy > 1.5 (boundary docs): "
      f"{(doc_entropy > 1.5).sum()} ({(doc_entropy > 1.5).mean()*100:.1f}%)")

Entropy statistics:
  Mean:   0.0266
  Median: 0.0000
  Max:    1.0932
  Min:    0.0000

Documents with entropy > 1.5 (boundary docs): 0 (0.0%)


---
## 6. Cluster Analysis & Fingerprinting

For each cluster we compute:
1. **Dominant documents** — 5 docs with highest cluster probability
2. **Top TF-IDF keywords** — topical fingerprint of the cluster
3. **Category distribution** — how ground-truth labels distribute
4. **Boundary documents** — high-entropy docs near cluster edges

In [62]:
# ── Cluster fingerprinting ──────────────────────────────────────

for cluster_id in range(optimal_k):
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*70}")

    # 1. Dominant documents: highest prob[cluster_id]
    probs_for_cluster = cluster_probs[:, cluster_id]
    top5_idx = np.argsort(probs_for_cluster)[-5:][::-1]
    print(f"\n--- Top 5 Dominant Documents ---")
    for rank, idx in enumerate(top5_idx, 1):
        cat = label_names[cleaned_labels[idx]]
        prob = probs_for_cluster[idx]
        preview = cleaned_texts[idx][:150].replace('\n', ' ')
        print(f"  [{rank}] p={prob:.3f} | {cat} | {preview}...")

    # 2. Top TF-IDF keywords
    cluster_mask = hard_labels == cluster_id
    cluster_docs = [cleaned_texts[j] for j in range(len(cleaned_texts))
                    if cluster_mask[j]]
    if len(cluster_docs) >= 5:
        tfidf = TfidfVectorizer(max_features=15, stop_words='english')
        tfidf.fit(cluster_docs)
        keywords = tfidf.get_feature_names_out()
        print(f"\n--- Top Keywords: {', '.join(keywords)}")
    else:
        print(f"\n--- Too few documents ({len(cluster_docs)}) for keyword extraction")

    # 3. Category distribution for strongly-assigned docs (prob > 0.6)
    strong_mask = probs_for_cluster > 0.6
    strong_cats = [label_names[cleaned_labels[j]]
                   for j in range(len(cleaned_texts)) if strong_mask[j]]
    if strong_cats:
        cat_counts = Counter(strong_cats).most_common(5)
        print(f"\n--- Category Distribution (prob > 0.6, top 5):")
        for cat, cnt in cat_counts:
            print(f"      {cat:40s}: {cnt}")

    # 4. Boundary documents: high entropy + second-best cluster is this one
    sorted_probs = np.argsort(cluster_probs, axis=1)
    second_best = sorted_probs[:, -2]  # second highest cluster per doc
    boundary_mask = (second_best == cluster_id) & (doc_entropy > np.percentile(doc_entropy, 75))
    boundary_indices = np.where(boundary_mask)[0]
    if len(boundary_indices) > 0:
        # Sort by entropy descending
        boundary_sorted = boundary_indices[np.argsort(doc_entropy[boundary_indices])[-5:][::-1]]
        print(f"\n--- Boundary Documents (high entropy, 2nd-best = this cluster):")
        for idx in boundary_sorted:
            ent = doc_entropy[idx]
            top2_clusters = np.argsort(cluster_probs[idx])[-2:][::-1]
            top2_probs = cluster_probs[idx][top2_clusters]
            preview = cleaned_texts[idx][:120].replace('\n', ' ')
            print(f"    entropy={ent:.3f} | clusters={top2_clusters} "
                  f"p={top2_probs} | {preview}...")


CLUSTER 0

--- Top 5 Dominant Documents ---
  [1] p=1.000 | rec.sport.baseball | Can someone send me ticket ordering information for the following teams:  Baltimore, Philadelphia and Boston.  Also, if you have a home schedule avail...
  [2] p=1.000 | rec.sport.hockey | In article <1993Apr26.023650.16749@spang.Camosun.BC.CA>, ua256@freenet.Victoria.BC.CA (Tom Moffat) says:  And won't they have to change their name to ...
  [3] p=1.000 | rec.sport.hockey | I am sure some bashers of Pens fans are pretty confused about the lack of any kind of posts about the recent Pens massacre of the Devils. Actually, I ...
  [4] p=1.000 | rec.sport.hockey | In article <1r1chb$5l2@jethro.Corp.Sun.COM> jake@rambler.Eng.Sun.COM (Jason Cockroft) writes:      Well, I'm a Wings fan and I think the FIRST thing t...
  [5] p=1.000 | rec.sport.hockey | In article <93109.134719IO91748@MAINE.MAINE.EDU> Jon Carr <IO91748@MAINE.MAINE.EDU> writes: I don't know the exact coverage in the states.  In Canada ...

--- Top

In [63]:
# ── Cluster Overlap Heatmap ─────────────────────────────────────
# For each pair (i,j), compute average of min(prob[i], prob[j]) across
# all documents. This reveals which clusters bleed into each other.

overlap_matrix = np.zeros((optimal_k, optimal_k))
for i in range(optimal_k):
    for j in range(optimal_k):
        # Average of elementwise min across all documents
        overlap_matrix[i, j] = np.mean(
            np.minimum(cluster_probs[:, i], cluster_probs[:, j])
        )

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(overlap_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=range(optimal_k), yticklabels=range(optimal_k),
            ax=ax)
ax.set_title('Cluster Overlap Heatmap\n(avg min(p_i, p_j) across all documents)',
             fontsize=14)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
plt.tight_layout()
plt.savefig('cluster_overlap_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved cluster_overlap_heatmap.png")

Saved cluster_overlap_heatmap.png


In [64]:
# ── UMAP Scatter Plots ──────────────────────────────────────────

# Plot 1: Colored by dominant GMM cluster, sized by entropy
fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    umap_embeddings_2d[:, 0], umap_embeddings_2d[:, 1],
    c=hard_labels, cmap='tab20', alpha=0.5,
    s=5 + doc_entropy * 20  # High entropy → larger point
)
ax.set_title('UMAP 2D — Colored by GMM Cluster (size ∝ entropy)', fontsize=14)
ax.set_xlabel('UMAP-1', fontsize=12)
ax.set_ylabel('UMAP-2', fontsize=12)
plt.colorbar(scatter, ax=ax, label='Cluster ID')
plt.tight_layout()
plt.savefig('umap_visualization.png', bbox_inches='tight')
plt.show()
print("Saved umap_visualization.png")

# Plot 2: Colored by original 20newsgroups category
fig, ax = plt.subplots(figsize=(14, 10))
scatter2 = ax.scatter(
    umap_embeddings_2d[:, 0], umap_embeddings_2d[:, 1],
    c=cleaned_labels, cmap='tab20', alpha=0.5, s=3
)
ax.set_title('UMAP 2D — Colored by Original 20 Newsgroups Category', fontsize=14)
ax.set_xlabel('UMAP-1', fontsize=12)
ax.set_ylabel('UMAP-2', fontsize=12)
plt.colorbar(scatter2, ax=ax, label='Category ID')

# Add a legend mapping IDs to names
unique_labels = sorted(set(cleaned_labels))
legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=plt.cm.tab20(l / 20), markersize=6,
                   label=label_names[l])
                   for l in unique_labels]
ax.legend(handles=legend_elements, loc='center left',
          bbox_to_anchor=(1.15, 0.5), fontsize=7, ncol=1)
plt.tight_layout()
plt.savefig('umap_original_categories.png', bbox_inches='tight')
plt.show()
print("Saved umap_original_categories.png")

Saved umap_visualization.png
Saved umap_original_categories.png


### A Note on Cross-Topic Clusters

Some clusters may combine categories that appear unrelated (e.g., `talk.politics.mideast` +
`alt.atheism`). This is **not** header/metadata leakage — the cleaning pipeline strips all
email headers (From:, Organization:, NNTP-Posting-Host:, etc.) before embedding.

These cross-topic clusters reflect **genuine semantic overlap** in the 20 Newsgroups data:
- Middle East political posts frequently involve religious debates (Islam, Judaism, Christianity)
- Atheism posters on 1990s USENET regularly engaged in Middle East religious-political discussions
- Both groups used similar vocabulary: "belief", "evidence", "conflict", "fundamentalism"

This is exactly what fuzzy clustering is designed to capture — documents that belong to multiple
semantic communities simultaneously. The GMM soft assignments quantify this overlap rather
than forcing an artificial boundary.

---
## 7. Semantic Cache Implementation

### Design Decisions

**Why build from scratch instead of using Redis/Memcached?**
- Full control over cluster-partitioned lookup strategy
- No external service dependency — runs entirely in-process
- Can integrate GMM soft assignments directly into cache logic

**Architecture: Cluster-partitioned cache**
- Instead of a flat cache (O(n) lookup against all entries), entries are
  partitioned by their dominant cluster
- A new query only searches within its own cluster partition → O(n/k) lookup
- For boundary queries (high entropy), we search all relevant partitions
  to avoid false cache misses

**No external cache libraries used** — only Python built-ins (`dict`, `list`,
`defaultdict`) and `numpy` for cosine similarity math.

In [65]:
import numpy as np
import time
from collections import defaultdict


class SemanticCache:
    """
    Cluster-partitioned semantic cache.

    Instead of a flat cache (O(n) lookup), entries are partitioned by
    dominant cluster. A new query only searches within its own cluster
    partition, reducing lookup to O(n/k) where k = number of clusters.

    For boundary queries (high entropy, membership spread across 2+
    clusters), we search all relevant cluster partitions to avoid
    false misses.

    The similarity threshold is the single most important tunable
    parameter. See threshold_sweep_analysis() in Section 8 for its
    full behavioral analysis.
    """

    def __init__(self, threshold=0.85, n_clusters=None):
        self.threshold = threshold
        self.n_clusters = n_clusters

        # Core storage: cluster_id → list of cache entries
        # Each entry is a plain dict — no external data structures
        self.partitions = defaultdict(list)

        # Stats tracking for monitoring and analysis
        self.hit_count = 0
        self.miss_count = 0
        self.total_lookup_time = 0.0

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """
        Hand-written cosine similarity. No sklearn, no scipy.
        cos(theta) = (a . b) / (||a|| * ||b||)
        Returns float in [-1, 1]. Higher = more similar.
        """
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return float(np.dot(a, b) / (norm_a * norm_b))

    def _get_search_partitions(self, cluster_probs: np.ndarray,
                                entropy_threshold: float = 1.5) -> list:
        """
        Determine which cluster partitions to search.

        For low-entropy queries (clear cluster membership): search only
        the dominant cluster partition.

        For high-entropy queries (boundary/ambiguous): search all
        partitions where probability > 0.2. This prevents cache misses
        for semantically ambiguous queries.
        """
        from scipy.stats import entropy as _ent
        ent = _ent(cluster_probs)

        if ent > entropy_threshold:
            # Boundary query: search all clusters with meaningful membership
            return [i for i, p in enumerate(cluster_probs) if p > 0.2]
        else:
            # Clear query: search only dominant cluster
            return [int(np.argmax(cluster_probs))]

    def lookup(self, query_embedding: np.ndarray,
               cluster_probs: np.ndarray) -> tuple:
        """
        Search the cache for a semantically similar prior query.
        Returns: (entry_or_None, best_similarity_score, dominant_cluster)
        """
        start = time.time()
        dominant_cluster = int(np.argmax(cluster_probs))
        partitions_to_search = self._get_search_partitions(cluster_probs)

        best_score = -1.0
        best_entry = None

        for cluster_id in partitions_to_search:
            for entry in self.partitions[cluster_id]:
                score = self._cosine_similarity(
                    query_embedding,
                    entry["embedding"]
                )
                if score > best_score:
                    best_score = score
                    best_entry = entry

        self.total_lookup_time += time.time() - start

        if best_score >= self.threshold:
            self.hit_count += 1
            return best_entry, best_score, dominant_cluster

        self.miss_count += 1
        return None, best_score, dominant_cluster

    def store(self, query: str, embedding: np.ndarray, result: str,
              cluster_probs: np.ndarray) -> None:
        """
        Store a new query-result pair in the appropriate cluster partition.
        We store under the dominant cluster. For boundary queries, this
        means they'll be found by other boundary queries searching
        multiple partitions.
        """
        dominant_cluster = int(np.argmax(cluster_probs))
        entry = {
            "query": query,
            "embedding": embedding,
            "result": result,
            "dominant_cluster": dominant_cluster,
            "cluster_distribution": cluster_probs.tolist(),
            "timestamp": time.time(),
            "access_count": 0
        }
        self.partitions[dominant_cluster].append(entry)

    def flush(self) -> None:
        """Wipe all cache entries and reset stats."""
        self.partitions = defaultdict(list)
        self.hit_count = 0
        self.miss_count = 0
        self.total_lookup_time = 0.0

    def get_stats(self) -> dict:
        total_queries = self.hit_count + self.miss_count
        total_entries = sum(len(v) for v in self.partitions.values())
        return {
            "total_entries": total_entries,
            "hit_count": self.hit_count,
            "miss_count": self.miss_count,
            "hit_rate": round(self.hit_count / total_queries, 3)
                        if total_queries > 0 else 0.0,
            "avg_lookup_time_ms": round(
                (self.total_lookup_time / total_queries * 1000), 3
            ) if total_queries > 0 else 0.0,
            "partition_sizes": {
                k: len(v) for k, v in self.partitions.items()
            }
        }

    def get_all_entries(self) -> list:
        """Return all cache entries for inspection endpoint."""
        all_entries = []
        for cluster_id, entries in self.partitions.items():
            for entry in entries:
                all_entries.append({
                    "query": entry["query"],
                    "dominant_cluster": entry["dominant_cluster"],
                    "cluster_distribution": entry["cluster_distribution"],
                    "timestamp": entry["timestamp"]
                })
        return all_entries


# Quick sanity test
cache = SemanticCache(threshold=0.85, n_clusters=optimal_k)
print(f"SemanticCache initialized with threshold=0.85, k={optimal_k}")
print(f"Stats: {cache.get_stats()}")

SemanticCache initialized with threshold=0.85, k=10
Stats: {'total_entries': 0, 'hit_count': 0, 'miss_count': 0, 'hit_rate': 0.0, 'avg_lookup_time_ms': 0.0, 'partition_sizes': {}}


---
## 8. Threshold Sweep Analysis

The similarity threshold controls the trade-off between:
- **Low threshold** → high hit rate but many wrong-topic results (false positives)
- **High threshold** → precise results but cache is rarely useful (low hit rate)

We sweep from 0.60 to 0.99 and measure hit rate, precision, and false positive
rate to find the optimal operating point.

In [66]:
# ── Create test data for threshold sweep ────────────────────────
# We create seed queries (populate the cache) and test queries
# (check if cache returns correct results) per topic.

# Seed queries: representative questions for each of the 20 categories
seed_queries_by_topic = {
    'alt.atheism': [
        'arguments against the existence of god',
        'why atheism is a rational worldview',
    ],
    'comp.graphics': [
        'how to render 3D graphics efficiently',
        'best algorithms for ray tracing',
    ],
    'comp.os.ms-windows.misc': [
        'fixing Windows driver issues',
        'Windows operating system configuration',
    ],
    'comp.sys.ibm.pc.hardware': [
        'best IBM PC compatible motherboards',
        'PC hardware upgrade recommendations',
    ],
    'comp.sys.mac.hardware': [
        'Macintosh hardware specifications',
        'Apple Mac computer components',
    ],
    'comp.windows.x': [
        'X Window System configuration',
        'configuring X11 display server',
    ],
    'misc.forsale': [
        'items for sale used electronics',
        'selling computer parts cheap',
    ],
    'rec.autos': [
        'best cars for daily driving',
        'automobile engine performance tuning',
    ],
    'rec.motorcycles': [
        'motorcycle riding safety tips',
        'best motorcycles for beginners',
    ],
    'rec.sport.baseball': [
        'baseball statistics and player rankings',
        'MLB season predictions and analysis',
    ],
    'rec.sport.hockey': [
        'NHL hockey team standings',
        'ice hockey game strategy and tactics',
    ],
    'sci.crypt': [
        'public key cryptography algorithms',
        'encryption and data security methods',
    ],
    'sci.electronics': [
        'electronic circuit design basics',
        'building amplifier circuits',
    ],
    'sci.med': [
        'medical treatment for chronic diseases',
        'latest medical research findings',
    ],
    'sci.space': [
        'NASA space exploration missions',
        'orbital mechanics and satellite launches',
    ],
    'soc.religion.christian': [
        'Christian theology and biblical interpretation',
        'church community and faith discussions',
    ],
    'talk.politics.guns': [
        'gun control legislation debate',
        'second amendment rights discussion',
    ],
    'talk.politics.mideast': [
        'Middle East political conflicts',
        'geopolitics in the Middle East region',
    ],
    'talk.politics.misc': [
        'general political discussion and opinions',
        'government policy debates',
    ],
    'talk.religion.misc': [
        'comparative religion discussions',
        'philosophical aspects of religion',
    ],
}

# Test queries: similar but differently phrased, with known correct topic
test_queries = [
    ('does god exist philosophical debate', 'alt.atheism'),
    ('computer graphics rendering pipeline', 'comp.graphics'),
    ('Windows OS troubleshooting tips', 'comp.os.ms-windows.misc'),
    ('IBM compatible PC upgrades', 'comp.sys.ibm.pc.hardware'),
    ('Apple Macintosh hardware repairs', 'comp.sys.mac.hardware'),
    ('X11 window manager setup', 'comp.windows.x'),
    ('selling used computer equipment', 'misc.forsale'),
    ('car maintenance and repairs', 'rec.autos'),
    ('motorcycle gear recommendations', 'rec.motorcycles'),
    ('baseball world series predictions', 'rec.sport.baseball'),
    ('hockey playoff bracket analysis', 'rec.sport.hockey'),
    ('RSA encryption key generation', 'sci.crypt'),
    ('building radio frequency circuits', 'sci.electronics'),
    ('symptoms and treatment of flu', 'sci.med'),
    ('space shuttle mission history', 'sci.space'),
    ('Bible study and interpretation', 'soc.religion.christian'),
    ('firearms regulation policy', 'talk.politics.guns'),
    ('Israel Palestine conflict analysis', 'talk.politics.mideast'),
    ('tax policy and government spending', 'talk.politics.misc'),
    ('world religions comparison', 'talk.religion.misc'),
    ('proof that god does not exist', 'alt.atheism'),
    ('3D modeling and animation software', 'comp.graphics'),
    ('Windows blue screen crash fix', 'comp.os.ms-windows.misc'),
    ('PC motherboard and CPU compatibility', 'comp.sys.ibm.pc.hardware'),
    ('Mac computer memory upgrade', 'comp.sys.mac.hardware'),
    ('X Window display resolution settings', 'comp.windows.x'),
    ('garage sale electronics deals', 'misc.forsale'),
    ('best fuel efficient vehicles', 'rec.autos'),
    ('Harley Davidson motorcycle reviews', 'rec.motorcycles'),
    ('MLB pitcher ERA statistics', 'rec.sport.baseball'),
    ('NHL goalie save percentage', 'rec.sport.hockey'),
    ('symmetric vs asymmetric encryption', 'sci.crypt'),
    ('soldering electronic components', 'sci.electronics'),
    ('cancer treatment research advances', 'sci.med'),
    ('Mars rover exploration discoveries', 'sci.space'),
    ('Christian prayer and worship', 'soc.religion.christian'),
    ('concealed carry permit laws', 'talk.politics.guns'),
    ('Middle East peace negotiations', 'talk.politics.mideast'),
    ('healthcare reform political debate', 'talk.politics.misc'),
    ('meaning of life in different religions', 'talk.religion.misc'),
    ('arguments for atheism', 'alt.atheism'),
    ('image processing pixel manipulation', 'comp.graphics'),
    ('Windows registry editing guide', 'comp.os.ms-windows.misc'),
    ('hard drive installation for PCs', 'comp.sys.ibm.pc.hardware'),
    ('Macintosh printer compatibility', 'comp.sys.mac.hardware'),
    ('configuring Xterm terminal emulator', 'comp.windows.x'),
    ('buy and sell used hardware', 'misc.forsale'),
    ('vintage classic car restoration', 'rec.autos'),
    ('sport bike vs cruiser comparison', 'rec.motorcycles'),
    ('home run records in baseball history', 'rec.sport.baseball'),
]

print(f"Seed queries: {sum(len(v) for v in seed_queries_by_topic.values())}")
print(f"Test queries: {len(test_queries)}")

Seed queries: 40
Test queries: 50


In [67]:
# ── Threshold Sweep ─────────────────────────────────────────────
# GMM is now fit on raw embeddings, so we predict_proba directly
# on raw embeddings — no UMAP transform needed.

gmm_for_sweep = gmm  # Already fitted on raw embeddings

# Pre-embed all queries (seed + test)
all_seed_queries = []
all_seed_topics = []
for topic, queries in seed_queries_by_topic.items():
    for q in queries:
        all_seed_queries.append(q)
        all_seed_topics.append(topic)

seed_embeddings = embedder.encode(all_seed_queries, normalize_embeddings=True)
test_query_texts = [q for q, _ in test_queries]
test_query_topics = [t for _, t in test_queries]
test_embeddings = embedder.encode(test_query_texts, normalize_embeddings=True)

# Get cluster probs directly from raw embeddings (no UMAP needed)
seed_cluster_probs = gmm_for_sweep.predict_proba(seed_embeddings)
test_cluster_probs = gmm_for_sweep.predict_proba(test_embeddings)

# Finer threshold steps (0.025) for a smooth tradeoff curve
thresholds = np.arange(0.50, 1.0, 0.025)
hit_rates = []
precisions = []
false_positive_rates = []

for t in tqdm(thresholds, desc="Threshold sweep"):
    # Fresh cache for each threshold
    sweep_cache = SemanticCache(threshold=t, n_clusters=optimal_k)

    # Populate with seed queries
    for i, (q, topic) in enumerate(zip(all_seed_queries, all_seed_topics)):
        sweep_cache.store(
            query=q,
            embedding=seed_embeddings[i],
            result=f"Result for topic: {topic}",
            cluster_probs=seed_cluster_probs[i]
        )

    # Test all test queries
    hits = 0
    correct_hits = 0
    false_positives = 0

    for i, (q_text, expected_topic) in enumerate(test_queries):
        entry, score, _ = sweep_cache.lookup(
            test_embeddings[i], test_cluster_probs[i]
        )
        if entry is not None:
            hits += 1
            if expected_topic in entry["result"]:
                correct_hits += 1
            else:
                false_positives += 1

    total = len(test_queries)
    hit_rate = hits / total
    precision = correct_hits / hits if hits > 0 else 0.0
    fpr = false_positives / total

    hit_rates.append(hit_rate)
    precisions.append(precision)
    false_positive_rates.append(fpr)

    print(f"  t={t:.3f} | hit_rate={hit_rate:.3f} | "
          f"precision={precision:.3f} | FPR={fpr:.3f}")

print("\nSweep complete.")

Threshold sweep:   0%|          | 0/20 [00:00<?, ?it/s]

  t=0.500 | hit_rate=0.640 | precision=0.969 | FPR=0.020
  t=0.525 | hit_rate=0.600 | precision=0.967 | FPR=0.020
  t=0.550 | hit_rate=0.540 | precision=1.000 | FPR=0.000
  t=0.575 | hit_rate=0.460 | precision=1.000 | FPR=0.000
  t=0.600 | hit_rate=0.420 | precision=1.000 | FPR=0.000


Threshold sweep:  75%|███████▌  | 15/20 [00:00<00:00, 72.59it/s]

  t=0.625 | hit_rate=0.400 | precision=1.000 | FPR=0.000
  t=0.650 | hit_rate=0.360 | precision=1.000 | FPR=0.000
  t=0.675 | hit_rate=0.280 | precision=1.000 | FPR=0.000
  t=0.700 | hit_rate=0.160 | precision=1.000 | FPR=0.000
  t=0.725 | hit_rate=0.120 | precision=1.000 | FPR=0.000
  t=0.750 | hit_rate=0.100 | precision=1.000 | FPR=0.000
  t=0.775 | hit_rate=0.040 | precision=1.000 | FPR=0.000
  t=0.800 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.825 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.850 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.875 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.900 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.925 | hit_rate=0.000 | precision=0.000 | FPR=0.000
  t=0.950 | hit_rate=0.000 | precision=0.000 | FPR=0.000


Threshold sweep: 100%|██████████| 20/20 [00:00<00:00, 74.65it/s]

  t=0.975 | hit_rate=0.000 | precision=0.000 | FPR=0.000

Sweep complete.


In [68]:
# ── Plot threshold analysis ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(thresholds, hit_rates, 'b-o', linewidth=2, label='Hit Rate', markersize=4)
ax.plot(thresholds, precisions, 'g-s', linewidth=2, label='Precision', markersize=4)
ax.plot(thresholds, false_positive_rates, 'r-^', linewidth=2,
        label='False Positive Rate', markersize=4)

# Find recommended threshold: among thresholds with hit_rate > 0.3,
# pick the most conservative (highest) one with maximum precision.
valid_mask = np.array(hit_rates) > 0.3
if valid_mask.any():
    valid_precisions = np.array(precisions)[valid_mask]
    max_prec = valid_precisions.max()
    # Among max-precision thresholds, choose the highest (most conservative)
    candidates = np.where(valid_mask)[0][valid_precisions == max_prec]
    best_idx = candidates[-1]  # highest threshold with max precision
    recommended_threshold = thresholds[best_idx]
else:
    # Fallback: highest threshold where hit_rate > 0.1
    fallback_mask = np.array(hit_rates) > 0.1
    if fallback_mask.any():
        best_idx = np.where(fallback_mask)[0][-1]  # Highest such threshold
        recommended_threshold = thresholds[best_idx]
    else:
        recommended_threshold = 0.80  # Safe default

ax.axvline(x=recommended_threshold, color='purple', linestyle='--',
           linewidth=2.5,
           label=f'Recommended threshold = {recommended_threshold:.3f}')

ax.set_xlabel('Similarity Threshold', fontsize=13)
ax.set_ylabel('Rate', fontsize=13)
ax.set_title('Threshold Sweep Analysis\nHit Rate vs Precision vs False Positive Rate',
             fontsize=14)
ax.legend(fontsize=11, loc='center right')
ax.set_xlim(0.48, 1.0)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', bbox_inches='tight')
plt.show()

print(f"\nRecommended threshold: {recommended_threshold:.3f}")
print(f"  At this threshold:")
idx = 0
for i, t in enumerate(thresholds):
    if abs(t - recommended_threshold) < 0.001:
        idx = i
        break
print(f"    Hit Rate:            {hit_rates[idx]:.3f}")
print(f"    Precision:           {precisions[idx]:.3f}")
print(f"    False Positive Rate: {false_positive_rates[idx]:.3f}")


Recommended threshold: 0.650
  At this threshold:
    Hit Rate:            0.360
    Precision:           1.000
    False Positive Rate: 0.000


### Threshold Selection Rationale

We select the threshold that **maximizes precision while maintaining a useful hit rate**
(above 30%). Below the chosen threshold, false positives rise sharply — the cache starts
returning results from wrong topics. Above it, cache utility collapses as it becomes
too strict to match even semantically identical queries.

The precision-recall curve reveals the fundamental tension: a cache that helps often
(high hit rate) but gives wrong answers is worse than one that helps rarely but is always
correct. We optimize for **correctness first, utility second**.

---
## 9. FastAPI Service (app.py)

The FastAPI app loads all pre-computed artifacts and exposes endpoints for:
- `POST /query` — semantic search with cache integration
- `GET /cache/stats` — cache performance metrics
- `DELETE /cache` — flush the cache
- `GET /cache/inspect` — view all cache entries (debug)
- `GET /health` — health check

The app is written to `app.py` by the cell below so it can be launched
independently with `uvicorn app:app --host 0.0.0.0 --port 8000`.

In [69]:
%%writefile app.py
# app.py - Semantic Search FastAPI Service
# Start with: uvicorn app:app --host 0.0.0.0 --port 8000
#
# This file is auto-generated by the main.ipynb notebook.
# It loads pre-computed artifacts (embeddings, models, ChromaDB)
# and provides a REST API for semantic search with caching.

from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import numpy as np
import time
import pickle
import chromadb
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from scipy.stats import entropy as scipy_entropy
from typing import Optional


# ══════════════════════════════════════════════════════════════════
# Semantic Cache — hand-written, no external cache libraries
# ══════════════════════════════════════════════════════════════════

class SemanticCache:
    """
    Cluster-partitioned semantic cache.
    Entries are partitioned by dominant cluster for O(n/k) lookup.
    Boundary queries (high entropy) search multiple partitions.
    """

    def __init__(self, threshold=0.85, n_clusters=None):
        self.threshold = threshold
        self.n_clusters = n_clusters
        self.partitions = defaultdict(list)
        self.hit_count = 0
        self.miss_count = 0
        self.total_lookup_time = 0.0

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Hand-written cosine similarity using only numpy."""
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return float(np.dot(a, b) / (norm_a * norm_b))

    def _get_search_partitions(self, cluster_probs: np.ndarray,
                                entropy_threshold: float = 1.5) -> list:
        """Search dominant partition for clear queries, multiple for ambiguous."""
        ent = scipy_entropy(cluster_probs)
        if ent > entropy_threshold:
            return [i for i, p in enumerate(cluster_probs) if p > 0.2]
        else:
            return [int(np.argmax(cluster_probs))]

    def lookup(self, query_embedding: np.ndarray,
               cluster_probs: np.ndarray) -> tuple:
        """Search cache for semantically similar prior query."""
        start = time.time()
        dominant_cluster = int(np.argmax(cluster_probs))
        partitions_to_search = self._get_search_partitions(cluster_probs)

        best_score = -1.0
        best_entry = None

        for cluster_id in partitions_to_search:
            for entry in self.partitions[cluster_id]:
                score = self._cosine_similarity(
                    query_embedding, entry["embedding"]
                )
                if score > best_score:
                    best_score = score
                    best_entry = entry

        self.total_lookup_time += time.time() - start

        if best_score >= self.threshold:
            self.hit_count += 1
            return best_entry, best_score, dominant_cluster

        self.miss_count += 1
        return None, best_score, dominant_cluster

    def store(self, query: str, embedding: np.ndarray, result: str,
              cluster_probs: np.ndarray) -> None:
        """Store query-result pair in the dominant cluster partition."""
        dominant_cluster = int(np.argmax(cluster_probs))
        entry = {
            "query": query,
            "embedding": embedding,
            "result": result,
            "dominant_cluster": dominant_cluster,
            "cluster_distribution": cluster_probs.tolist(),
            "timestamp": time.time(),
            "access_count": 0
        }
        self.partitions[dominant_cluster].append(entry)

    def flush(self) -> None:
        """Wipe all cache entries and reset stats."""
        self.partitions = defaultdict(list)
        self.hit_count = 0
        self.miss_count = 0
        self.total_lookup_time = 0.0

    def get_stats(self) -> dict:
        total_queries = self.hit_count + self.miss_count
        total_entries = sum(len(v) for v in self.partitions.values())
        return {
            "total_entries": total_entries,
            "hit_count": self.hit_count,
            "miss_count": self.miss_count,
            "hit_rate": round(self.hit_count / total_queries, 3)
                        if total_queries > 0 else 0.0,
            "avg_lookup_time_ms": round(
                (self.total_lookup_time / total_queries * 1000), 3
            ) if total_queries > 0 else 0.0,
            "partition_sizes": {
                k: len(v) for k, v in self.partitions.items()
            }
        }

    def get_all_entries(self) -> list:
        """Return all cache entries for inspection."""
        all_entries = []
        for cluster_id, entries in self.partitions.items():
            for entry in entries:
                all_entries.append({
                    "query": entry["query"],
                    "dominant_cluster": entry["dominant_cluster"],
                    "cluster_distribution": entry["cluster_distribution"],
                    "timestamp": entry["timestamp"]
                })
        return all_entries


# ══════════════════════════════════════════════════════════════════
# Load pre-computed artifacts
# ══════════════════════════════════════════════════════════════════

print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading GMM model...")
with open('gmm.pkl', 'rb') as f:
    gmm = pickle.load(f)

# NOTE: GMM is fit on raw 384-dim embeddings (not UMAP).
# Query embeddings go directly to gmm.predict_proba() — no UMAP step needed.

print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection("newsgroups_collection")
print(f"ChromaDB collection loaded: {collection.count()} documents")

# Initialize semantic cache
cache = SemanticCache(threshold=0.85)

# ══════════════════════════════════════════════════════════════════
# FastAPI Application
# ══════════════════════════════════════════════════════════════════

app = FastAPI(
    title="Semantic Search API",
    description="20 Newsgroups semantic search with fuzzy clustering cache",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Timing Middleware ────────────────────────────────────────────
@app.middleware("http")
async def add_timing_header(request, call_next):
    """Add X-Response-Time-Ms header to every response."""
    start = time.time()
    response = await call_next(request)
    elapsed_ms = round((time.time() - start) * 1000, 2)
    response.headers["X-Response-Time-Ms"] = str(elapsed_ms)
    return response


# ── Request/Response Models ──────────────────────────────────────
class QueryRequest(BaseModel):
    query: str


class QueryResponse(BaseModel):
    query: str
    cache_hit: bool
    matched_query: Optional[str]
    similarity_score: float
    result: str
    dominant_cluster: int


# ── Helper: compute result on cache miss ─────────────────────────
def compute_result(query_embedding, n_results=5):
    """
    On cache miss, query ChromaDB for top-5 most similar documents.
    Returns a formatted string result.
    """
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    formatted = []
    for i, (doc, meta) in enumerate(zip(docs, metas)):
        formatted.append(
            f"[Result {i+1}] Category: {meta.get('category', 'unknown')}\n"
            f"{doc[:300]}..."
        )
    return "\n\n".join(formatted)


# ── Endpoint 1: POST /query ──────────────────────────────────────
@app.post("/query", response_model=QueryResponse)
async def query_endpoint(body: QueryRequest,
                          background_tasks: BackgroundTasks):
    """Semantic search with cache. Returns top-5 similar documents."""
    query_text = body.query.strip()
    if not query_text:
        raise HTTPException(status_code=400, detail="Query cannot be empty")

    # Embed the query using the same model as document embeddings
    query_embedding = embedder.encode(query_text, normalize_embeddings=True)

    # Get cluster membership directly from raw embedding (no UMAP needed)
    cluster_probs = gmm.predict_proba([query_embedding])[0]
    dominant_cluster = int(np.argmax(cluster_probs))

    # Check semantic cache first
    cached_entry, similarity_score, _ = cache.lookup(
        query_embedding, cluster_probs
    )

    if cached_entry is not None:
        return QueryResponse(
            query=query_text,
            cache_hit=True,
            matched_query=cached_entry["query"],
            similarity_score=round(similarity_score, 4),
            result=cached_entry["result"],
            dominant_cluster=dominant_cluster
        )

    # Cache miss — compute result from ChromaDB
    result = compute_result(query_embedding)

    # Store in cache via background task (non-blocking)
    background_tasks.add_task(
        cache.store, query_text, query_embedding, result, cluster_probs
    )

    return QueryResponse(
        query=query_text,
        cache_hit=False,
        matched_query=None,
        similarity_score=round(similarity_score, 4),
        result=result,
        dominant_cluster=dominant_cluster
    )


# ── Endpoint 2: GET /cache/stats ─────────────────────────────────
@app.get("/cache/stats")
async def cache_stats():
    """Return cache performance metrics."""
    return cache.get_stats()


# ── Endpoint 3: DELETE /cache ────────────────────────────────────
@app.delete("/cache")
async def flush_cache():
    """Flush all cache entries and reset stats."""
    cache.flush()
    return {"message": "Cache flushed successfully", "status": "ok"}


# ── Endpoint 4: GET /cache/inspect ───────────────────────────────
@app.get("/cache/inspect")
async def inspect_cache():
    """Debug endpoint: view full cache contents and partition sizes."""
    return {
        "entries": cache.get_all_entries(),
        "partition_sizes": {
            k: len(v) for k, v in cache.partitions.items()
        }
    }


# ── Health Check ─────────────────────────────────────────────────
@app.get("/health")
async def health():
    """Health check endpoint."""
    return {
        "status": "healthy",
        "cache_entries": sum(len(v) for v in cache.partitions.values()),
        "chromadb_docs": collection.count()
    }

Overwriting app.py


In [70]:
# ── Verify app.py ────────────────────────────────────────────────
import os, py_compile, re

fsize = os.path.getsize('app.py')
print(f"app.py size: {fsize:,} bytes")

# Syntax check — raises py_compile.PyCompileError on failure
py_compile.compile('app.py', doraise=True)
print("Syntax check: PASSED")

# Ensure no UMAP imports or code remain (comments mentioning UMAP are fine)
with open('app.py', 'r', encoding='utf-8') as f:
    for lineno, line in enumerate(f, 1):
        stripped = line.split('#')[0]  # ignore comments
        if 'umap' in stripped.lower():
            raise AssertionError(f"app.py line {lineno} has UMAP code: {line.strip()}")
print("UMAP-free check: PASSED (no UMAP imports or code)")
print(f"\nAll app.py verification checks passed  ({fsize:,} bytes, valid syntax, no UMAP dep)")

app.py size: 12,340 bytes
Syntax check: PASSED
UMAP-free check: PASSED (no UMAP imports or code)

All app.py verification checks passed  (12,340 bytes, valid syntax, no UMAP dep)


---
## 10. Artifact Saving & Final Summary

In [71]:
# ── Save all model artifacts ────────────────────────────────────

# Save UMAP model (50-dim, fitted on training embeddings)
with open('umap_model.pkl', 'wb') as f:
    pickle.dump(umap_50d, f)
print("Saved umap_model.pkl")

# Save GMM model (already saved above, but ensure latest version)
with open('gmm.pkl', 'wb') as f:
    pickle.dump(gmm, f)
print("Saved gmm.pkl")

# Save label names for the API to reference
with open('label_names.json', 'w') as f:
    json.dump(label_names, f)
print("Saved label_names.json")

print("\n" + "="*60)
print("ALL ARTIFACTS SAVED SUCCESSFULLY")
print("="*60)
print("\nFiles produced:")
expected_files = [
    'main.ipynb', 'app.py', 'requirements.txt',
    'embeddings.npy', 'cluster_probs.npy',
    'gmm.pkl', 'umap_model.pkl', 'label_names.json',
    'cluster_selection_bic_aic.png', 'cluster_selection_silhouette.png',
    'cluster_selection_coherence.png', 'umap_visualization.png',
    'umap_original_categories.png', 'cluster_overlap_heatmap.png',
    'threshold_analysis.png'
]
for f_name in expected_files:
    exists = os.path.exists(f_name)
    status = 'OK' if exists else 'MISSING'
    print(f"  [{status}] {f_name}")

# Check ChromaDB directory
chroma_exists = os.path.isdir('./chroma_db')
print(f"  [{'OK' if chroma_exists else 'MISSING'}] chroma_db/")

print(f"\nTo start the API server, run:")
print(f"  uvicorn app:app --host 0.0.0.0 --port 8000")

Saved umap_model.pkl
Saved gmm.pkl
Saved label_names.json

ALL ARTIFACTS SAVED SUCCESSFULLY

Files produced:
  [OK] main.ipynb
  [OK] app.py
  [OK] requirements.txt
  [OK] embeddings.npy
  [OK] cluster_probs.npy
  [OK] gmm.pkl
  [OK] umap_model.pkl
  [OK] label_names.json
  [OK] cluster_selection_bic_aic.png
  [OK] cluster_selection_silhouette.png
  [OK] cluster_selection_coherence.png
  [OK] umap_visualization.png
  [OK] umap_original_categories.png
  [OK] cluster_overlap_heatmap.png
  [OK] threshold_analysis.png
  [OK] chroma_db/

To start the API server, run:
  uvicorn app:app --host 0.0.0.0 --port 8000
